# 导入相关包

In [ ]:
!pip install opencc
!pip install pyLDAvis

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
from collections import Counter
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt
import pyLDAvis.gensim_models as gensimvis #可視化
import pyLDAvis #可視化
import json
import opencc # 簡轉繁用
import jieba
import re
import os

# CKIP要的東西

In [ ]:
!pip install ckiptagger
!pip install ckipnlp
!pip install ckip_classic

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from ckipnlp.pipeline import CkipPipeline, CkipDocument

pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER

In [ ]:
ws = WS("/content/drive/MyDrive/CKIP/data")
pos = POS("/content/drive/MyDrive/CKIP/data")
ner = NER("/content/drive/MyDrive/CKIP/data")

# 停用詞list

In [ ]:
import json
# nuk停用頂端.json (用在斷詞前)
with open("/content/drive/MyDrive/collab-topic/停用部分句/nuk停用頂端.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTonuk=[]
for word in stopwords:
  RemoveTonuk=set(word.get("stopwords", []))# 用set防止重複的東西
# print("toRemove:\n",toRemove)

# nuch停用頂端.json (用在斷詞前)
with open("/content/drive/MyDrive/collab-topic/停用部分句/nchu停用頂端.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTonuchu=[]
for word in stopwords:
  RemoveTonuchu=set(word.get("stopwords", []))# 用set防止重複的東西

# 無主題式詞 (用在特徵選取後)
with open("/content/drive/MyDrive/collab-topic/停用部分句/無主題式詞.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTheMeaningless=[]
for word in stopwords:
  RemoveTheMeaningless=set(word.get("stopwords", []))# 用set防止重複的東西
# 只准許的詞性
valid_pos = {"Na", "Nb", "Nc", "Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VE", "VF", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}
BanSthwithEntity_Types=['PERSON','LOC','GPE']
HFW_rank=10 # High Frequency Word only rank to 10 = 關鍵字 = (經特徵選取後的)特徵字

# 讀取檔案並合併

most_common[-1]表示most_common列表的最後一個元素，也就是第10高的詞。[-1][1]表示取這個元素的第二個元素，也就是這個詞出現的次數。

## nkust

In [ ]:
# nkust
# 讀取 JSON 檔案
from collections import Counter
import unicodedata
import json
import re
with open("/content/drive/MyDrive/collab-topic/斷詞前的資料/已有斷詞_程式不碰/nkust-topic.json", "r") as f:
    data = json.load(f)
# destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/rank10_banPersonName/nkust(Rank10)_personTest.json'
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/rank10_banPersonNameAndLOC_GPE/nkust.json'
# BanSthwithEntity_Types=['PERSON','LOC','GPE']
# valid_pos = {"Na", "Nb", "Nc", "Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VE", "VF", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}
# HFW_rank=10 # High Frequency Word only rank to 10 
nkustJsonFile=[]
# 對 JSON 檔案內的文字進行斷詞
counterFor=0
for item in data:
    # if counterFor==1:  break
    counterFor=counterFor+1
    # 取得文字內容
    text = ' '.join( item.get("title", None) )
    # print("text1\n",text)
    text = text+' '.join( item.get("content", None) )
    # print("text2\n",text)
    # print(text)
    # print("-----===============================------------------------",counterFor)
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)
    # CKIP 斷詞與詞性標註
    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
    # 作實體識別(用於判別人名)
    ner_text = pipeline.get_ner(doc)

    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        # if ner_text[i][j]=='PERSON':
        #   continue
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])

    # 禁人名出現在featureWord的清單(list)
    banSth=[] # PersonName or LOC or GPE
    # print("ner_text\n",ner_text)
    for i, sent_ner in enumerate(ner_text):
      # print("sent_ner\n",sent_ner)
      for j, ner in enumerate(sent_ner):
        # print("ner\n",ner)
        # print("ner[1]\n",ner[1])
        if ner[1] in BanSthwithEntity_Types:
          banSth.append(ner[0])
          # print("ner\n",ner)
    
    # 只留中文字
    # print("feature_words1",featureWord)
    newfeatureWord = []
    pattern = re.compile(r'[\u4e00-\u9fff]+')

    for word in featureWord:
      # 把列表中的每個單詞轉成字符串再進行匹配
      matched_words = pattern.findall(str(word))# 我覺得跟str有關
      # 把匹配到的單詞加入到新列表中
      newfeatureWord.extend(matched_words)
    
    # print("feature_words2",newfeatureWord)
    featureWord=newfeatureWord

    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
    # 取得前10個出現頻率最高的詞彙(一個名次多人至)
    counter = Counter(featureWord)
    most_common = counter.most_common(HFW_rank)
    New_common=[]
    tenth_count = most_common[-1][1]
    print("tenth_count:",tenth_count)
    for word, count in counter.most_common():
      if count >= tenth_count:
        New_common.append(word)
      else:
        break
    featureWord=New_common
    print(featureWord)
    
    # 禁在清單內的東西
    if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
      featureWord=[w for w in featureWord if w not in banSth]
    # 檢查關鍵字用
    if len(featureWord)>20:
      print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
    if len(featureWord)==0:
      continue
    nkustJsonFile.append({
        "TitleAndDESC":featureWord,
        "Topic":["教育活動"],
    })
#     # print("nkustJsonFile:\n",nkustJsonFile)
# with open(destination, 'w', encoding='utf-8') as f:
#     json.dump(nkustJsonFile, f, ensure_ascii=False, indent=4)
# print(len(nkustJsonFile)) # 269

In [ ]:

print("banPerson\n",banPerson)
print("len(banPerson)",len(banPerson))
print("ner_text\n",ner_text)

## nuk

In [ ]:
#nuk
from collections import Counter
import json
import re

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞前的資料/已有斷詞_程式不碰/nuk.json", "r") as f:
    data = json.load(f)
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/nuk.json'

nukJsonFile=[]
counterFor=0
# 對 JSON 檔案內的文字進行斷詞
for item in data:
    # if counterFor==10: break
    counterFor+=1
    # 取得文字內容
    text = ' '.join(item.get("title", []))
    # print("title\n",text)
    content=item.get("content", [])
    # for s in toRemove: # 這個也可以刪除停用詞
    #     while s in content:
    #         content.remove(s)
    
    # 和停用句json檔比對
    # 如果content內容有部分完全和停用句檔內一模一樣，那麼刪除content內容中有和停用句檔內一模一樣的內容
    content = [c for c in content if c not in RemoveTonuk]
    # print("content\n",content)
    text =text+' '.join(content)
    # print("結合text\n",text)

    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)
    # CKIP 斷詞與詞性標註
    # print("text:\n"text)
    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
    # 作實體識別(用於判別人名)
    ner_text = pipeline.get_ner(doc)

    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        # if ner_text[i][j]=='PERSON':
        #   continue
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])

    # 禁人名出現在featureWord的清單(list)
    banSth=[] # PersonName or LOC or GPE
    # print("ner_text\n",ner_text)
    for i, sent_ner in enumerate(ner_text):
      # print("sent_ner\n",sent_ner)
      for j, ner in enumerate(sent_ner):
        # print("ner\n",ner)
        # print("ner[1]\n",ner[1])
        if ner[1] in BanSthwithEntity_Types:
          banSth.append(ner[0])
          # print("ner\n",ner)
    
    # 只留中文字
    # print("feature_words1",featureWord)
    newfeatureWord = []
    pattern = re.compile(r'[\u4e00-\u9fff]+')

    for word in featureWord:
      # 把列表中的每個單詞轉成字符串再進行匹配
      matched_words = pattern.findall(str(word))# 我覺得跟str有關
      # 把匹配到的單詞加入到新列表中
      newfeatureWord.extend(matched_words)
    
    # print("feature_words2",newfeatureWord)
    featureWord=newfeatureWord
    
    # 去低頻(1)詞
    counter = Counter(featureWord)
    if len(counter):# 裡面有東西才執行，不然就是空文本
      New_common=[]
      LowerBound_count = 2
      for word, count in counter.most_common():
        if count < LowerBound_count:  break
        New_common.append(word)
      featureWord=New_common
    else:# 空文本(無效文本)
      continue
    # print(featureWord)
    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
    
    # 禁在清單內的東西
    if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
      featureWord=[w for w in featureWord if w not in banSth]
    # 檢查關鍵字用
    if len(featureWord)>20:
      print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
    
    nukJsonFile.append({
        "TitleAndDESC":featureWord,
        "Topic":["教育活動"],
    })
    # print(featureWord)

with open(destination, 'w', encoding='utf-8') as f:
    json.dump(nukJsonFile, f, ensure_ascii=False, indent=4)
print(len(nukJsonFile)) # 215

## 開放政府

In [ ]:
# 開放政府
import json
import glob

# 取得資料夾內所有的 JSON 檔案路徑
json_files = glob.glob('/content/drive/MyDrive/collab-topic/斷詞前的資料/開放政府_複製/*.json 的副本')
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/govFromOpenData.json'

gov_json = []

# 逐一讀取 JSON 檔案
for json_file in json_files:
    # print(json_file)
    with open(json_file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        counterFor=0
        for item in json_data:
          # if counterFor==10:  break
          counterFor+=1
          text = ' '.join(item.get("Title", []))
          
          Description=item.get("Description", [])
          if Description != None and Description !=[]: # 防止加空(因為加None會error)
              text = text+' '.join(Description)
          # DealBefore=text
          Topics = item.get("Topic", None)
          # 對Topics kry裡面放list or str 做處理
          if type(Topics) == str :
            Topics = [Topics]
          # print("text:\n",text)
          
          # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
          text = re.sub(r'[a-zA-Z]', ' ', text)
          # CKIP 斷詞與詞性標註
          doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
          ws_text=pipeline.get_ws(doc)
          pos_text=pipeline.get_pos(doc)
          # 作實體識別(用於判別人名)
          ner_text = pipeline.get_ner(doc)

          # 只選特定詞性
          featureWord=[]
          for i,sent_p in enumerate(pos_text):
            for j,word_p in enumerate(sent_p):
              # if ner_text[i][j]=='PERSON':
              #   continue
              if word_p in valid_pos:
                featureWord.append(ws_text[i][j])

          # 禁人名出現在featureWord的清單(list)
          banSth=[] # PersonName or LOC or GPE
          # print("ner_text\n",ner_text)
          for i, sent_ner in enumerate(ner_text):
            # print("sent_ner\n",sent_ner)
            for j, ner in enumerate(sent_ner):
              # print("ner\n",ner)
              # print("ner[1]\n",ner[1])
              if ner[1] in BanSthwithEntity_Types:
                banSth.append(ner[0])
                # print("ner\n",ner)
          
          # 只留中文字
          # print("feature_words1",featureWord)
          newfeatureWord = []
          pattern = re.compile(r'[\u4e00-\u9fff]+')
          for word in featureWord:
            # 把列表中的每個單詞轉成字符串再進行匹配
            matched_words = pattern.findall(str(word))# 我覺得跟str有關
            # 把匹配到的單詞加入到新列表中
            newfeatureWord.extend(matched_words)
          
          # print("feature_words2",newfeatureWord)
          featureWord=newfeatureWord
          
          # 去低頻(1)詞
          counter = Counter(featureWord)
          if len(counter):# 裡面有東西才執行，不然就是空文本
            New_common=[]
            LowerBound_count = 2
            for word, count in counter.most_common():
              if count < LowerBound_count:  break
              New_common.append(word)
            featureWord=New_common
          else:# 空文本(無效文本)
            # print("Error")
            # print(f"In {counterFor} of {json_file} have empty")
            continue

          # 刪除無意義詞(無主題式)-停用詞
          featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
          
          # 禁在清單內的東西
          if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
            featureWord=[w for w in featureWord if w not in banSth]
          # 檢查關鍵字用
          if len(featureWord)>20:
            print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
          if len(featureWord)<=2:
            continue
          gov_json.append({
                    "TitleAndDESC":featureWord,
                    "Topic":Topics,
                })
          # print("featureWord:\n",featureWord)

# 將新的 JSON 資料存入檔案中。
with open(destination, 'w', encoding='utf-8') as f:
    json.dump(gov_json, f, ensure_ascii=False, indent=4)
print(len(gov_json)) # 1190

## nchu

In [ ]:
# nchu
source='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/nchu-topic'
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/nchu.json'
json_files = glob.glob(source+'/*.json')
newJsonFile = []
counterFor=0
# 逐一讀取 JSON 檔案
for json_file in json_files:
  # print(json_files) #1012
  # print(json_file) #1012
  # if counterFor==10:  break
  counterFor=counterFor+1
  with open(json_file, 'r', encoding='utf-8') as f:
      json_data = json.load(f)
      # print('json_data:\n',json_data)
      # 取得文字內容
      text = ' '.join( json_data.get("title", []) )
      # print('title:\n',text)
      content_texts = json_data.get("content", [])
      # print('content_text_s(原):\n',content_texts)
      # 刪除nuhu頂端固定格式-停用詞
      for AstopWord in RemoveTonuchu:
        content_texts = [text.replace(AstopWord, ' ') for text in content_texts]

      for content_text in content_texts:
        text = text + ' ' + content_text
      # print('article text:\n',text)
      # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
      text = re.sub(r'[a-zA-Z]', ' ', text)
      # CKIP 斷詞與詞性標註
      doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
      ws_text=pipeline.get_ws(doc)
      pos_text=pipeline.get_pos(doc)
      # 作實體識別(用於判別人名)
      ner_text = pipeline.get_ner(doc)
      # 只選特定詞性
      featureWord=[]
      for i,sent_p in enumerate(pos_text):
        for j,word_p in enumerate(sent_p):
          # if ner_text[i][j]=='PERSON':
          #   continue
          if word_p in valid_pos:
            featureWord.append(ws_text[i][j])
      # 禁人名/LOC/GPE出現在featureWord的清單(list)
      banSth=[] # PersonName or LOC or GPE
      # print("ner_text\n",ner_text)
      for i, sent_ner in enumerate(ner_text):
        # print("sent_ner\n",sent_ner)
        for j, ner in enumerate(sent_ner):
          # print("ner\n",ner)
          # print("ner[1]\n",ner[1])
          if ner[1] in BanSthwithEntity_Types:
            banSth.append(ner[0])
            # print("ner\n",ner)
      # 只留中文字
      newfeatureWord = []
      pattern = re.compile(r'[\u4e00-\u9fff]+')
      for word in featureWord:
        # 把列表中的每個單詞轉成字符串再進行匹配
        matched_words = pattern.findall(str(word))# 我覺得跟str有關
        # 把匹配到的單詞加入到新列表中
        newfeatureWord.extend(matched_words)
      featureWord=newfeatureWord
      # 去低頻(1)詞
      counter = Counter(featureWord)
      if len(counter):# 裡面有東西才執行，不然就是空文本
        New_common=[]
        LowerBound_count = 2
        for word, count in counter.most_common():
          if count < LowerBound_count:  break
          New_common.append(word)
        featureWord=New_common
      else:# 空文本(無效文本)
        # print(f"In {counterFor} of {json_file} featureWord is Error",)
        continue
      # 刪除無意義詞(無主題式)-停用詞
      featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
      # 禁在清單內的東西
      if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
        featureWord=[w for w in featureWord if w not in banSth]
      # 檢查關鍵字用
      if len(featureWord)>20:
        print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
      if len(featureWord)<=2: continue
      newJsonFile.append({
          "TitleAndDESC":featureWord,
          "Topic":["教育活動"],
      })
      # print(featureWord)

with open(destination, 'w', encoding='utf-8') as f:
  json.dump(newJsonFile, f, ensure_ascii=False, indent=4)
print(len(newJsonFile)) # 3050

## ntnu

In [ ]:
# ntnu (入快速資料夾fun)
source='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/ntnu-topic'
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/rank10_banPersonNameAndLOC_GPE/ntnu.json'
json_files = glob.glob(source+'/*.json')
newJsonFile = []
counter=0
# 逐一讀取 JSON 檔案
for json_file in json_files:
  # if counter==1:  break
  # counter=counter+1
  with open(json_file, 'r', encoding='utf-8') as f:
      json_data = json.load(f)
      # print('json_data:\n',json_data)
      # 取得文字內容
      text = ' '.join( json_data.get("title", []) )
      # print('title:\n',text)
      content_texts = json_data.get("content", [])
      # print('content_text_s:\n',content_texts)
      for content_text in content_texts:
        # print('content_text:\n',content_text)
        text = text + ' ' + content_text
      # print('article text:\n',text)
      # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
      text = re.sub(r'[a-zA-Z]', ' ', text)
      # CKIP 斷詞與詞性標註
      doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
      ws_text=pipeline.get_ws(doc)
      pos_text=pipeline.get_pos(doc)
      # 作實體識別(用於判別人名)
      ner_text = pipeline.get_ner(doc)
      # 只選特定詞性
      featureWord=[]
      for i,sent_p in enumerate(pos_text):
        for j,word_p in enumerate(sent_p):
          # if ner_text[i][j]=='PERSON':
          #   continue
          if word_p in valid_pos:
            featureWord.append(ws_text[i][j])
      # 禁人名/LOC/GPE出現在featureWord的清單(list)
      banSth=[] # PersonName or LOC or GPE
      # print("ner_text\n",ner_text)
      for i, sent_ner in enumerate(ner_text):
        # print("sent_ner\n",sent_ner)
        for j, ner in enumerate(sent_ner):
          # print("ner\n",ner)
          # print("ner[1]\n",ner[1])
          if ner[1] in BanSthwithEntity_Types:
            banSth.append(ner[0])
            # print("ner\n",ner)
      # 只留中文字
      newfeatureWord = []
      pattern = re.compile(r'[\u4e00-\u9fff]+')
      for word in featureWord:
        # 把列表中的每個單詞轉成字符串再進行匹配
        matched_words = pattern.findall(str(word))# 我覺得跟str有關
        # 把匹配到的單詞加入到新列表中
        newfeatureWord.extend(matched_words)
      featureWord=newfeatureWord
      # 取得前10個出現頻率最高的詞彙(一個名次多人至)
      counter = Counter(featureWord)
      most_common = counter.most_common(HFW_rank)
      if len(most_common):# 裡面有東西才執行，不然就是空文本
        New_common=[]
        tenth_count = most_common[-1][1]
        for word, count in counter.most_common():
          if count >= tenth_count:
            New_common.append(word)
          else:
            break
        featureWord=New_common
      # 刪除無意義詞(無主題式)-停用詞
      featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
      # 禁在清單內的東西
      if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
        featureWord=[w for w in featureWord if w not in banSth]
      # 檢查關鍵字用
      if len(featureWord)>20:
        print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
      # 無意義文本不放
      if len(featureWord)==0:
        print(f"In {counterFor} of {json_file} featureWord is Error",)
        continue
      newJsonFile.append({
          "TitleAndDESC":featureWord,
          "Topic":["教育活動"],
      })
      print(featureWord)

with open(destination, 'w', encoding='utf-8') as f:
  json.dump(newJsonFile, f, ensure_ascii=False, indent=4)
print(len(newJsonFile)) # 226

## 資料夾快速fun

In [ ]:
# source='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/nchu-topic'
# destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/rank10_banPersonNameAndLOC_GPE/nchu.json'
def featureWordsFromFolder(source,destination):# source=Folder is include many document, destination= is a document
  json_files = glob.glob(source+'/*.json')
  newJsonFile = []
  counterFor=0
  # 逐一讀取 JSON 檔案
  for json_file in json_files:
    # print(json_files) 
    # print(json_file) 
    # if counterFor==10:  break # 測試
    counterFor=counterFor+1
    with open(json_file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        # print('json_data:\n',json_data)
        # 取得文字內容
        text = ' '.join( json_data.get("title", []) )
        # print('title:\n',text)
        content_texts = json_data.get("content", [])
        # print('content_text_s(原):\n',content_texts)

        # # 刪除nuhu頂端固定格式-停用詞
        # for AstopWord in RemoveTonuchu:
        #   content_texts = [text.replace(AstopWord, ' ') for text in content_texts]

        for content_text in content_texts:
          text = text + ' ' + content_text
        # print('article text:\n',text)
        # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
        text = re.sub(r'[a-zA-Z]', ' ', text)
        # CKIP 斷詞與詞性標註
        doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
        ws_text=pipeline.get_ws(doc)
        pos_text=pipeline.get_pos(doc)
        # 作實體識別(用於判別人名)
        ner_text = pipeline.get_ner(doc)
        # 只選特定詞性
        featureWord=[]
        for i,sent_p in enumerate(pos_text):
          for j,word_p in enumerate(sent_p):
            # if ner_text[i][j]=='PERSON':
            #   continue
            if word_p in valid_pos:
              featureWord.append(ws_text[i][j])
        # 禁人名/LOC/GPE出現在featureWord的清單(list)
        banSth=[] # PersonName or LOC or GPE
        # print("ner_text\n",ner_text)
        for i, sent_ner in enumerate(ner_text):
          # print("sent_ner\n",sent_ner)
          for j, ner in enumerate(sent_ner):
            # print("ner\n",ner)
            # print("ner[1]\n",ner[1])
            if ner[1] in BanSthwithEntity_Types:
              banSth.append(ner[0])
              # print("ner\n",ner)
        # 只留中文字
        newfeatureWord = []
        pattern = re.compile(r'[\u4e00-\u9fff]+')
        for word in featureWord:
          # 把列表中的每個單詞轉成字符串再進行匹配
          matched_words = pattern.findall(str(word))# 我覺得跟str有關
          # 把匹配到的單詞加入到新列表中
          newfeatureWord.extend(matched_words)
        featureWord=newfeatureWord
        # 去低頻(1)詞
        counter = Counter(featureWord)
        if len(counter):# 裡面有東西才執行，不然就是空文本
          New_common=[]
          LowerBound_count = 2
          for word, count in counter.most_common():
            if count < LowerBound_count:  break
            New_common.append(word)
          featureWord=New_common
        else:# 空文本(無效文本)
          continue
        # 刪除無意義詞(無主題式)-停用詞
        featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
        # 禁在清單內的東西
        if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
          featureWord=[w for w in featureWord if w not in banSth]
        # 檢查關鍵字用
        if len(featureWord)>20:
          print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
        # 無意義文本不放
        if len(featureWord)<=2: continue
        
        newJsonFile.append({
            "TitleAndDESC":featureWord,
            "Topic":["教育活動"],
        })
        # print(featureWord)

  with open(destination, 'w', encoding='utf-8') as f:
    json.dump(newJsonFile, f, ensure_ascii=False, indent=4)
  print(len(newJsonFile))
  return

## 20230228 NewDataSet

In [ ]:
PreDestination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/'
# ntnu
featureWordsFromFolder('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/20230228NewDataSet/NewDataSet/#nkust-topic',PreDestination+'ntnu'+'.json')
# ntust
featureWordsFromFolder('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/20230228NewDataSet/NewDataSet/ntust-news',PreDestination+'ntust'+'.json')
# nyust
featureWordsFromFolder('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/20230228NewDataSet/NewDataSet/nyust-news',PreDestination+'nyust'+'.json')
# 0228nkust
featureWordsFromFolder('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/0228nkust-topic',PreDestination+'0228nkust'+'.json')

## 合併成一個json(nuk+ntnu+nuchu+govOpenData-4950)+0228nkust+ntust+nyust

In [ ]:
import json
import glob
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/nuk+0228nkust+govFromOpenData+nchu+ntnu+0228nkust+ntust+nyust.json'

# 取得資料夾內所有的 JSON 檔案路徑
json_files = glob.glob('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/*.json')
merged_json = []

# 逐一讀取 JSON 檔案
for json_file in json_files:
    print(json_files)
    with open(json_file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        for item in json_data:
            TitleAndDESC = item.get("TitleAndDESC", None)
            Topic = item.get("Topic", None)
            if TitleAndDESC==None:
              print("TitleAndDESC_error")
            # if Topic==None:
            #   print("Topic_error")
            
            merged_json.append({
                    "TitleAndDESC":TitleAndDESC,
                    "Topic":Topic,
                })
# +nkust看看，然後數有多少key or 元素
# 將新的 JSON 資料存入檔案中。
with open(destination, 'w', encoding='utf-8') as f:
    json.dump(merged_json, f, ensure_ascii=False, indent=4)
print("總資料量：",len(merged_json)) # 4950 # 6784

## 獨立停用詞和同義詞

In [ ]:
import json
import re

synonymsDocument="/content/drive/MyDrive/collab-topic/停用部分句/synonyms.json"
BeScanJson='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/nuk+0228nkust+govFromOpenData+nchu+ntnu+0228nkust+ntust+nyust.json'
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/nuk+0228nkust+govFromOpenData+nchu+ntnu+ntust+nyust(newFeatureSelect).json'
# 讀取同義詞JSON檔案
with open(synonymsDocument, 'r', encoding='utf-8') as f:
    synonyms = json.load(f)

# 預編譯正則表達式
replace_pattern = re.compile(r'^(?=.{2,}).*盃$')
remove_pattern = re.compile(r'^(?!会|人|金枝演)(^.+社$)$')

# 讀取資料集檔案
with open(BeScanJson, 'r') as f:
    data = json.load(f)

# 使用集合來加速判斷
# RemoveTheMeaningless = set(['停用詞1', '停用詞2', ...])

# 替換同義詞
for item in data:
    item_text = ' '.join(item['TitleAndDESC'])
    for key, values in synonyms.items():
        for value in values:
            item_text = item_text.replace(value, key)
    item['TitleAndDESC'] = item_text.split()
    # print("item['TitleAndDESC']0\n",item['TitleAndDESC'])

    # 刪除無意義詞(無主題式)-停用詞
    item['TitleAndDESC'] = [c for c in item['TitleAndDESC'] if c not in RemoveTheMeaningless]

    # 使用預編譯正則表達式
    item['TitleAndDESC'] = [replace_pattern.sub('比賽', c) for c in item['TitleAndDESC']]
    item['TitleAndDESC'] = [remove_pattern.sub('社團', c) for c in item['TitleAndDESC']]
    # print("final\n", item['TitleAndDESC'])
with open(destination, 'w', encoding='utf-8') as f:
  json.dump(data, f, ensure_ascii=False, indent=4)


# lda 超參數

In [ ]:
def outputToTxt(content,d):# 變量內容, 目的地：用於將變量輸出到文字檔裡面觀察
  with open(d, "w") as f:
    f.write(str(content))
  return

In [ ]:
# 用來創(看)變數-texts、dictionary、corpus
import json
from gensim import corpora, models
source='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/removeLowFrequencyWords/nuk+0228nkust+govFromOpenData+nchu+ntnu+ntust+nyust(newFeatureSelect).json'

# 讀取 JSON 檔案
with open(source, 'r') as f:
    data = json.load(f)

# 將詞彙列表合併為字串
docs = [" ".join(d["TitleAndDESC"]) for d in data]
# print("docs:\n",docs)
# outputToTxt(docs,'docs.txt')

# 創建詞袋
texts = [[word for word in doc.split()] for doc in docs]
# print("texts:\n",texts)
dictionary = corpora.Dictionary(texts)
# outputToTxt(texts,'texts.txt')
# print("dictionary:\n",dictionary)
# outputToTxt(dictionary,'dictionary.txt')
corpus = [dictionary.doc2bow(text) for text in texts]
# print("corpus:\n",corpus)
# outputToTxt(corpus,'corpus.txt')

# 創建LDA模型
# lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=10)

## 網格搜索 

In [ ]:
# 單進程版
from gensim.models.coherencemodel import CoherenceModel
import itertools
import numpy as np
from gensim.models import LdaModel
from gensim import corpora, models
# from multiprocessing import Pool, set_start_method

# 定义超参数空间
param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],# 6
        # 'alpha': [0.01, 0.1],#預設auto，會自動設置合適的
        # 'eta': [0.01, 0.1],
        'iterations': [300],
        'passes': [300]}

# 枚举所有超参数组合
param_list = list(itertools.product(*param_grid.values()))

# 定义模型评分函数
def evaluate_model(param):
    lda_model = LdaModel(corpus, id2word=dictionary, random_state=42, **dict(zip(param_grid.keys(), param)))
    coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
    coherence = coherence_model.get_coherence()
    perplex = lda_model.log_perplexity(corpus)# corpus_sparse改corpus
    return coherence, perplex

# 计算所有超参数组合的模型评分
scores = np.array([evaluate_model(param) for param in param_list])
# scores = np.array(pool.map(evaluate_model, param_list))
# 找到最优超参数组合
# best_param = param_list[scores.argmax()]
# print(f'Best param: {dict(zip(param_grid.keys(), best_param))}, score: {scores.max()}')
# 找到最优超参数组合
best_index = scores[:,0].argmax()# 从 scores 数组中选取所有行的第一列数据，然后返回最大值所在的索引。
best_param = param_list[best_index]
best_coherence = scores[best_index][0]
best_perplexity = scores[best_index][1] # 對於best_coherence而言的perplexity
print(f'Best param: {dict(zip(param_grid.keys(), best_param))}, coherence: {best_coherence}, perplexity: {best_perplexity}')

## (網格搜)統計

In [ ]:
import matplotlib.pyplot as plt

print("scores:\n",scores)
# 一致性折線圖
# 將一致性取出
coherence = [s[0] for s in scores]

# 設定 x 軸為評分的 index
x = range(2,11)#range(len(scores))

# 繪製折線圖(越大越好)
plt.plot(x, coherence, '-o')

# 設定圖片標題和軸標籤
plt.title('Coherence Scores')
plt.xlabel('Index')
plt.ylabel('Coherence')

# 顯示圖片
plt.show()

# 困惑度折線圖(越小越好)
# 將一致性取出
perplexity = [s[1] for s in scores]

# 設定 x 軸為評分的 index
x = range(2,11)#range(len(scores))

# 繪製折線圖
plt.plot(x, perplexity, '-o')

# 設定圖片標題和軸標籤
plt.title('perplexity Scores')
plt.xlabel('Index')
plt.ylabel('perplexity')

# 顯示圖片
plt.show()

param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],
        'iterations': [100],
        'passes': [100]}
scores:
 [[ 0.41621816 -8.25636009]
 [ 0.47640855 -8.20135025]
 [ 0.47584959 -8.19902545]# 最高一致性 次高困惑度
 [ 0.47032878 -8.27044123]# 次高一致性, 最高(最爛)困惑度
 [ 0.3950044  -8.34511019]# 第三高一致性 約倒數第三四困惑度
 [ 0.38528891 -8.39901253]
 [ 0.4489874  -8.40512712]
 [ 0.40466168 -8.43360969]
 [ 0.38551242 -8.45484986]]


---


Best param: {'num_topics': 4, 'iterations': 300, 'passes': 300}, coherence: 0.48269515386387024, perplexity: -8.196291530128656
 scores:
 [[ 0.42098567 -8.25166996]
 [ 0.47006016 -8.19652039]# 第三高一致性 最高(最爛)困惑度(1,2名差不多)
 [ 0.48269515 -8.19629153]# 最高一致性 最高(最爛)困惑度(1,2名差不多)
 [ 0.47035707 -8.28432034]# 次高一致性 倒數第三大(爛)困惑度
 [ 0.42619649 -8.34341665]
 [ 0.39830502 -8.40863981]
 [ 0.45917194 -8.41407957]# 第4高一致性 困惑度第三好(小)
 [ 0.4213375  -8.4492195 ]
 [ 0.39158982 -8.46941654]]

## 訓練模型

In [ ]:
param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],# 6
        # 'alpha': [0.01, 0.1],#預設auto，會自動設置合適的
        # 'eta': [0.01, 0.1],
        'iterations': [300],
        'passes': [300]}

In [ ]:
from gensim.models import LdaModel

# 训练主题模型
# manual
num_topics=5
iterations=300
passes=300
manual__param=(num_topics,iterations,passes)
lda_model = LdaModel(corpus, id2word=dictionary, random_state=42, **dict(zip(param_grid.keys(), manual__param)))
# lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, iterations=iterations,passes=passes, random_state=42)
# print(**dict(zip(param_grid.keys(), best_param)))
# print( best_param)
# print( param_grid.keys())
# auto best_param
# lda_model = LdaModel(corpus, id2word=dictionary, random_state=42, **dict(zip(param_grid.keys(), best_param)))



## 可視化(下載模型)

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
# 創建可視化資料
# auto
PSparameters=str(dict(zip(param_grid.keys(), manual__param)))
# PSparameters=str(dict(zip(param_grid.keys(), best_param)))
# manual_param=(3,300,300)
# PSparameters=str(dict(zip(param_grid.keys(), manual_param)))
vis_data = gensimvis.prepare(topic_model=lda_model, corpus=corpus, dictionary=dictionary)


# 将数据保存到JSON文件

pyLDAvis.save_json(vis_data, '/content/drive/MyDrive/collab-topic/主題模型實驗結果/0302/0302(NewFeatureSelection)'+PSparameters+'.json' )
# 顯示可視化
pyLDAvis.display(vis_data)

# 保存為HTML文件
pyLDAvis.save_html(vis_data, '/content/drive/MyDrive/collab-topic/主題模型實驗結果/0302/0302(NewFeatureSelection)'+PSparameters+'.html')
# 保存模型
lda_model.save('/content/drive/MyDrive/collab-topic/主題模型實驗結果/0302/0302(NewFeatureSelection)'+PSparameters+'.gensim')

## 主題內容(字)

In [ ]:
def outputToTxt(content,d):# 變量內容, 目的地：用於將變量輸出到文字檔裡面觀察
  with open(d, "w") as f:
    f.write(str(content))
  return

In [ ]:
# # 加載訓練好的主題模型
# model_name = "/content/drive/MyDrive/collab-topic/主題模型實驗結果/0302/0302(NewFeatureSelection){'num_topics': 3, 'iterations': 300, 'passes': 300}.gensim"
# # model = api.load(model_name)
# model = LdaModel.load(model_name)
# lda_model = model 

# topicContent=lda_model.print_topics(num_topics=best_param[0], num_words=100)
topicContent=lda_model.print_topics(num_topics=manual__param[0], num_words=100)
outputToTxt(topicContent,'/content/drive/MyDrive/collab-topic/主題模型實驗結果/0302/0302topicContent(NewFeatureSelection)'+PSparameters+'.txt')

# 預測新文本

In [ ]:
import re
from collections import Counter
from ckipnlp.pipeline import CkipPipeline, CkipDocument
# pipeline = CkipPipeline(opts={'con_parser': {'username': "<CKIP_USERNAME>", 'password': "<CKIP_PASSWORD>"}})
# pipeline = CkipPipeline()  # create a CKIP pipeline object
def preprocess_text_rank10(text):
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)

    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
     # 作實體識別(用於判別人名)
    ner_text = pipeline.get_ner(doc)
    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        # if ner_text[i][j]=='PERSON':
        #   continue
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])
    # 禁人名/LOC/GPE出現在featureWord的清單(list)
    banSth=[] # PersonName or LOC or GPE
    # print("ner_text\n",ner_text)
    for i, sent_ner in enumerate(ner_text):
      # print("sent_ner\n",sent_ner)
      for j, ner in enumerate(sent_ner):
        # print("ner\n",ner)
        # print("ner[1]\n",ner[1])
        if ner[1] in BanSthwithEntity_Types:
          banSth.append(ner[0])
          # print("ner\n",ner)
    # 只留中文字
    newfeatureWord = []
    pattern = re.compile(r'[\u4e00-\u9fff]+')
    for word in featureWord:
      # 把列表中的每個單詞轉成字符串再進行匹配
      matched_words = pattern.findall(str(word))# 我覺得跟str有關
      # 把匹配到的單詞加入到新列表中
      newfeatureWord.extend(matched_words)
    featureWord=newfeatureWord
    # 取得前10個出現頻率最高的詞彙(一個名次多人至)
    counter = Counter(featureWord)
    most_common = counter.most_common(HFW_rank)
    if len(most_common):# 裡面有東西才執行，不然就是空文本
      New_common=[]
      tenth_count = most_common[-1][1]
      for word, count in counter.most_common():
        if count >= tenth_count:
          New_common.append(word)
        else:
          break
      featureWord=New_common
    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
    # 禁在清單內的東西
    if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
      featureWord=[w for w in featureWord if w not in banSth]
    # 檢查關鍵字用
    if len(featureWord)>20:
      print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
    # 無意義文本不放
    if len(featureWord)==0:
      # print(f"In {counterFor} of {json_file} featureWord is Error",)
      print(f"Error: featureWord is empty",)
    return featureWord
    
new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動'
print(new_text)

# 預處理新文本，建立詞彙表
new_doc = preprocess_text_rank10(new_text)
print(new_doc)

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from collections import Counter

# pipeline = CkipPipeline(opts={'con_parser': {'username': "<CKIP_USERNAME>", 'password': "<CKIP_PASSWORD>"}})
# pipeline = CkipPipeline()  # create a CKIP pipeline object
def preprocess_text(text):
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)

    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
     # 作實體識別(用於判別人名)
    ner_text = pipeline.get_ner(doc)
    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        # if ner_text[i][j]=='PERSON':
        #   continue
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])
    # 禁人名/LOC/GPE出現在featureWord的清單(list)
    banSth=[] # PersonName or LOC or GPE
    # print("ner_text\n",ner_text)
    for i, sent_ner in enumerate(ner_text):
      # print("sent_ner\n",sent_ner)
      for j, ner in enumerate(sent_ner):
        # print("ner\n",ner)
        # print("ner[1]\n",ner[1])
        if ner[1] in BanSthwithEntity_Types:
          banSth.append(ner[0])
          # print("ner\n",ner)
    # 只留中文字
    newfeatureWord = []
    pattern = re.compile(r'[\u4e00-\u9fff]+')
    for word in featureWord:
      # 把列表中的每個單詞轉成字符串再進行匹配
      matched_words = pattern.findall(str(word))# 我覺得跟str有關
      # 把匹配到的單詞加入到新列表中
      newfeatureWord.extend(matched_words)
    featureWord=newfeatureWord
    # # 取得前10個出現頻率最高的詞彙(一個名次多人至)
    # counter = Counter(featureWord)
    # most_common = counter.most_common(HFW_rank)
    # if len(most_common):# 裡面有東西才執行，不然就是空文本
    #   New_common=[]
    #   tenth_count = most_common[-1][1]
    #   for word, count in counter.most_common():
    #     if count >= tenth_count:
    #       New_common.append(word)
    #     else:
    #       break
    #   featureWord=New_common
    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
    # 禁在清單內的東西
    if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
      featureWord=[w for w in featureWord if w not in banSth]
    # 檢查關鍵字用
    if len(featureWord)>20:
      print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
    # 無意義文本不放
    if len(featureWord)==0:
      # print(f"In {counterFor} of {json_file} featureWord is Error",)
      print(f"Error: featureWord is empty",)
    return featureWord
    
new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動'
print(new_text)

# 預處理新文本，建立詞彙表
new_doc = preprocess_text(new_text)
print(new_doc)

In [ ]:
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import gensim.downloader as api
from gensim.models.ldamodel import LdaModel

# # 加載訓練好的主題模型
# model_name = "/content/drive/MyDrive/collab-topic/主題模型實驗結果/0302/0302(NewFeatureSelection){'num_topics': 3, 'iterations': 300, 'passes': 300}.gensim"
# # model = api.load(model_name)
# model = LdaModel.load(model_name)
# 不載入
model = lda_model
# 創建新文本
new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動不管怎樣早睡早起都好吧'
print(new_text)

# 預處理新文本，建立詞彙表
new_doc = preprocess_text(new_text)
print(new_doc)
# dictionary = Dictionary.load(train_dictionary)
# dictionary = train_dictionary

# 將新文本轉換為詞袋表示
new_doc_bow = dictionary.doc2bow(new_doc)

# 預測新文本的主題
doc_topics = model.get_document_topics(new_doc_bow)

# 輸出主題預測結果
print(doc_topics)